1. การเตรียมข้อมูล (Data Preparation)
ส่วนนี้เป็นการสร้างชุดข้อมูลความต้องการ Personal Computer ตามสไลด์หน้า 9

In [18]:
import pandas as pd
import numpy as np

# ข้อมูลความต้องการสินค้า (A_t)
data_pc = {
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    't': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'A_t': [37, 40, 41, 37, 45, 50, 43, 47, 56, 52, 55, 54]
}
df = pd.DataFrame(data_pc)
print("--- ตารางข้อมูลตั้งต้น (PC Demand) ---")
print(df)

--- ตารางข้อมูลตั้งต้น (PC Demand) ---
   Month   t  A_t
0    Jan   1   37
1    Feb   2   40
2    Mar   3   41
3    Apr   4   37
4    May   5   45
5    Jun   6   50
6    Jul   7   43
7    Aug   8   47
8    Sep   9   56
9    Oct  10   52
10   Nov  11   55
11   Dec  12   54


2. Simple Moving Average (SMA)ตัวอย่าง: สไลด์หน้า 9 (คำนวณค่าพยากรณ์ $n=3$) 2สูตร: $F_{t} = \frac{\sum A_{t-i}}{n}$

In [19]:
# --- กำหนดตัวแปร ---
n = 3

# --- การคำนวณ ---
# ใช้ rolling mean และ shift(1) เพราะพยากรณ์เดือนปัจจุบัน ต้องใช้ข้อมูลเดือน "ก่อนหน้า"
df['SMA_3'] = df['A_t'].rolling(window=n).mean().shift(1)

print(f"\n--- Simple Moving Average (n={n}) ---")
print(df[['Month', 'A_t', 'SMA_3']])


--- Simple Moving Average (n=3) ---
   Month  A_t      SMA_3
0    Jan   37        NaN
1    Feb   40        NaN
2    Mar   41        NaN
3    Apr   37  39.333333
4    May   45  39.333333
5    Jun   50  41.000000
6    Jul   43  44.000000
7    Aug   47  46.000000
8    Sep   56  46.666667
9    Oct   52  48.666667
10   Nov   55  51.666667
11   Dec   54  54.333333


3. Weighted Moving Average (WMA)ตัวอย่าง: สไลด์หน้า 11 (พยากรณ์เดือน Nov) สูตร: $F_{t} = \sum W_{i}A_{t-i}$

In [20]:
# --- กำหนดตัวแปร ---
# ข้อมูลเดือน Aug, Sep, Oct เพื่อพยากรณ์ Nov
demands = [130, 110, 90]  # [Aug, Sep, Oct] -> [A_t-3, A_t-2, A_t-1]
weights = [0.17, 0.33, 0.50] # [W3, W2, W1] น้ำหนักเรียงจากไกลไปใกล้ (Aug->Oct)

# --- การคำนวณ ---
# F_11 = (0.17 * 130) + (0.33 * 110) + (0.50 * 90)
F_Nov = (weights[0] * demands[0]) + (weights[1] * demands[1]) + (weights[2] * demands[2])

print(f"\n--- Weighted Moving Average (Forecast for Nov) ---")
print(f"Calculation: ({weights[0]}*{demands[0]}) + ({weights[1]}*{demands[1]}) + ({weights[2]}*{demands[2]})")
print(f"Forecast Result: {F_Nov:.2f}")


--- Weighted Moving Average (Forecast for Nov) ---
Calculation: (0.17*130) + (0.33*110) + (0.5*90)
Forecast Result: 103.40


4. Exponential Smoothingตัวอย่าง: สไลด์หน้า 13 (Alpha = 0.5) 6สูตร: $F_{t} = F_{t-1} + \alpha(A_{t-1} - F_{t-1})$ 7

In [21]:
# --- กำหนดตัวแปร ---
alpha = 0.5
forecasts = [np.nan] * len(df) # สร้างลิสต์ว่าง

# --- เงื่อนไขเริ่มต้น (สไลด์หน้า 13) ---
# F_2 = A_1
forecasts[1] = df['A_t'][0]

# --- การคำนวณ (เริ่มตั้งแต่เดือน 3) ---
for t in range(2, len(df)):
    F_prev = forecasts[t-1]
    A_prev = df['A_t'][t-1]
    
    # สูตรหลัก
    F_new = F_prev + alpha * (A_prev - F_prev)
    forecasts[t] = F_new

df['Exp_Smooth'] = forecasts
print(f"\n--- Exponential Smoothing (Alpha={alpha}) ---")
print(df[['Month', 'A_t', 'Exp_Smooth']].head())


--- Exponential Smoothing (Alpha=0.5) ---
  Month  A_t  Exp_Smooth
0   Jan   37         NaN
1   Feb   40      37.000
2   Mar   41      38.500
3   Apr   37      39.750
4   May   45      38.375


5. Adjusted Exponential Smoothing (Trend)

ตัวอย่าง: สไลด์หน้า 15 (Alpha = 0.5, Beta = 0.3) สูตร: มี 3 ขั้นตอน (FWT, Level, Trend)

In [22]:
# --- กำหนดตัวแปร ---
alpha = 0.5
beta = 0.3
level = [np.nan] * len(df) # F_t ในสไลด์ (Level)
trend = [np.nan] * len(df) # T_t ในสไลด์ (Trend)
fwt = [np.nan] * len(df)   # FWT_t ในสไลด์ (Forecast)

# --- เงื่อนไขเริ่มต้น (สไลด์หน้า 15) ---
# เริ่มที่เดือน 2 (Feb): F_2 = A_1, T_2 = 0
level[1] = df['A_t'][0]  # 37
trend[1] = 0.0
fwt[1] = level[1] + trend[1] # Forecast เดือน Feb

# --- การคำนวณ (เริ่มเดือน 3) ---
for t in range(2, len(df)):
    A_prev = df['A_t'][t-1]   # ความต้องการจริงเดือนก่อน
    FWT_prev = fwt[t-1]       # ค่าพยากรณ์เดือนก่อน
    Trend_prev = trend[t-1]   # เทรนด์เดือนก่อน
    
    # 1. Update Level (F_t)
    # สูตร: F_t = FWT_{t-1} + alpha(A_{t-1} - FWT_{t-1})
    current_level = FWT_prev + alpha * (A_prev - FWT_prev)
    
    # 2. Update Trend (T_t)
    # สูตร: T_t = T_{t-1} + beta(F_t - FWT_{t-1})
    current_trend = Trend_prev + beta * (current_level - FWT_prev)
    
    # เก็บค่า
    level[t] = current_level
    trend[t] = current_trend
    
    # 3. Calculate Forecast (FWT_t)
    # สูตร: FWT_t = F_t + T_t
    fwt[t] = current_level + current_trend

df['Exp_Trend'] = fwt
print(f"\n--- Adjusted Exponential Smoothing ---")
print(df[['Month', 'A_t', 'Exp_Trend']].head())


--- Adjusted Exponential Smoothing ---
  Month  A_t  Exp_Trend
0   Jan   37        NaN
1   Feb   40  37.000000
2   Mar   41  38.950000
3   Apr   37  40.732500
4   May   45  39.063875


6. Linear Regressionตัวอย่าง: สไลด์หน้า 18 10สูตร: $b = \frac{\sum XY - n\bar{X}\bar{Y}}{\sum X^2 - n\bar{X}^2}$, $a = \bar{Y} - b\bar{X}$ 11

In [23]:
# --- กำหนดตัวแปร ---
X = df['t'].values
Y = df['A_t'].values
n = len(X)

# --- การคำนวณ ---
sum_x = np.sum(X)
sum_y = np.sum(Y)
sum_xy = np.sum(X * Y)
sum_x2 = np.sum(X**2)
mean_x = np.mean(X)
mean_y = np.mean(Y)

# คำนวณ b (Slope)
b = (sum_xy - n * mean_x * mean_y) / (sum_x2 - n * mean_x**2)

# คำนวณ a (Intercept)
a = mean_y - b * mean_x

print(f"\n--- Linear Regression Model ---")
print(f"Formula: Y = {a:.2f} + {b:.2f}X")
df['Linear_Reg'] = a + b * df['t']


--- Linear Regression Model ---
Formula: Y = 35.21 + 1.72X


7. Seasonality (Seasonal Index)ตัวอย่าง: สไลด์หน้า 19 (รถยนต์) สูตร: $S_i = \frac{D_i}{\sum D_k}$

In [24]:
# --- กำหนดตัวแปร ---
# ข้อมูลยอดจองรถ 3 ปี แยกรายไตรมาส
# แถว = ปี, คอลัมน์ = ไตรมาส 1, 2, 3, 4
data_seasonal = np.array([
    [12.6, 8.6, 6.3, 17.5],
    [14.1, 10.3, 18.2, 7.5],
    [10.6, 15.3, 8.1, 19.6]
])

# --- การคำนวณ ---
total_demand_all_years = np.sum(data_seasonal) # 148.7
total_demand_per_quarter = np.sum(data_seasonal, axis=0) # [Q1_sum, Q2_sum, Q3_sum, Q4_sum]

# คำนวณ Seasonal Index
seasonal_indices = total_demand_per_quarter / total_demand_all_years

print(f"\n--- Seasonal Indices ---")
for i, idx in enumerate(seasonal_indices):
    print(f"Quarter {i+1}: {idx:.2f}")


--- Seasonal Indices ---
Quarter 1: 0.25
Quarter 2: 0.23
Quarter 3: 0.22
Quarter 4: 0.30


8. Error Metrics (MAD, MSE, MAPD)

ตัวอย่าง: สไลด์หน้า 23 (ใช้ข้อมูล Moving Average) สูตร: ตามหน้า 22

In [25]:
# เลือกข้อมูลที่มีค่าทั้ง A_t และ Forecast (ตัดแถว NaN ทิ้ง)
df_eval = df.dropna(subset=['SMA_3']).copy()
actual = df_eval['A_t']
forecast = df_eval['SMA_3']

# --- การคำนวณ ---
error = actual - forecast
abs_error = np.abs(error)
sq_error = error ** 2
pct_error = np.abs(error / actual)

# หาค่าเฉลี่ย
MAD = np.mean(abs_error)
MSE = np.mean(sq_error)
MAPD = np.mean(pct_error) * 100

print(f"\n--- Error Metrics (SMA n=3) ---")
print(f"MAD: {MAD:.2f}")
print(f"MSE: {MSE:.2f}")
print(f"MAPD: {MAPD:.2f}%")


--- Error Metrics (SMA n=3) ---
MAD: 3.93
MSE: 25.56
MAPD: 7.90%


9. Tracking Signalตัวอย่าง: สไลด์หน้า 30-31 (ใช้ Linear Regression) สูตร: $TS_t = \frac{\text{Cum Error}}{MAD_t}$ 

In [28]:
# ใช้ผลจาก Linear Regression
df['Error_LR'] = df['A_t'] - df['Linear_Reg']

# --- การคำนวณ ---
# 1. Cumulative Error (ผลบวกสะสมของ Error)
df['Cum_Error'] = df['Error_LR'].cumsum()

# 2. MAD_t (MAD แบบสะสม ณ เวลา t)
# ใช้ expanding().mean() ของค่า Absolute Error
df['Abs_Error_LR'] = df['Error_LR'].abs()
df['MAD_t'] = df['Abs_Error_LR'].expanding().mean()

# 3. Tracking Signal
df['Tracking_Signal'] = df['Cum_Error'] / df['MAD_t']

print(f"\n--- Tracking Signal (First 5 months) ---")
print(df[['Month', 'Cum_Error', 'MAD_t', 'Tracking_Signal']])


--- Tracking Signal (First 5 months) ---
   Month     Cum_Error     MAD_t  Tracking_Signal
0    Jan  6.410256e-02  0.064103     1.000000e+00
1    Feb  1.404429e+00  0.702214     2.000000e+00
2    Mar  2.020979e+00  0.673660     3.000000e+00
3    Apr -3.086247e+00  1.782051    -1.731851e+00
4    May -1.917249e+00  1.659441    -1.155359e+00
5    Jun  2.527972e+00  2.123737     1.190341e+00
6    Jul -1.750583e+00  2.431568    -7.199397e-01
7    Aug -3.752914e+00  2.377914    -1.578238e+00
8    Sep  1.520979e+00  2.699689     5.633904e-01
9    Oct  1.071096e+00  2.474709     4.328168e-01
10   Nov  1.897436e+00  2.324857     8.161517e-01
11   Dec  1.421085e-14  2.289239     6.207678e-15
